# Tahap 06 — Comparative Analysis: Manual Coding vs BERTopic

## Judul Project

**Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic**

## Tujuan Notebook

Notebook ini digunakan untuk membandingkan hasil **Manual Coding** dengan hasil **BERTopic**.

Tahap ini mencakup:

1. Membaca hasil final Manual Coding.
2. Membaca hasil BERTopic Modeling.
3. Menggabungkan kedua hasil berdasarkan `chunk_id` / `doc_id`.
4. Membuat tabulasi silang antara `manual_selective_theme` dan `bertopic_topic_id`.
5. Mengukur kesesuaian hasil manual dan hasil komputasional menggunakan metrik clustering.
6. Menentukan mapping dominan antara topik BERTopic dan tema manual.
7. Membuat tabel representatif dokumen per topik.
8. Membuat visualisasi perbandingan.
9. Menyimpan output untuk kebutuhan laporan akhir dan presentasi.

## Input Utama

```text
data/processed/manual_coding_final.csv
data/processed/bertopic_document_topics_with_manual_reference.csv
data/processed/bertopic_topic_info.csv
data/processed/bertopic_topic_keywords.csv
```

## Output Utama

```text
data/processed/comparative_analysis_manual_vs_bertopic.csv
reports/tables/comparative_crosstab_theme_topic.csv
reports/tables/comparative_crosstab_theme_topic_pct.csv
reports/tables/bertopic_topic_manual_theme_mapping.csv
reports/tables/manual_theme_bertopic_topic_mapping.csv
reports/tables/comparative_metrics_summary.csv
reports/tables/representative_documents_by_topic.csv
reports/tables/comparative_narrative_summary.csv
reports/tables/stage06_output_manifest.json
reports/figures/comparative_heatmap_manual_theme_vs_bertopic.html
reports/figures/comparative_stacked_bar_manual_theme_by_topic.html
reports/figures/manual_vs_bertopic_distribution.html
```

## Catatan Metodologis

Perbandingan ini tidak menggunakan metrik akurasi klasifikasi biasa, karena BERTopic adalah metode **unsupervised learning**.  
Metrik yang lebih tepat adalah:

- Normalized Mutual Information (NMI)
- Adjusted Mutual Information (AMI)
- Homogeneity
- Completeness
- V-measure

Metrik tersebut mengukur seberapa besar struktur cluster BERTopic memiliki hubungan dengan label tema manual.

## 1. Import Library

Notebook menggunakan Pandas untuk analisis tabular, Scikit-learn untuk metrik clustering, dan Plotly untuk visualisasi interaktif.

In [1]:
# ============================================================
# Import Library
# ============================================================

from pathlib import Path
from datetime import datetime
import importlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Library dasar berhasil di-import.")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Library dasar berhasil di-import.
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
Pandas version: 2.3.3
Numpy version: 2.2.6


## 2. Setup Path Project

Notebook mendeteksi root project berdasarkan file hasil Tahap 05.

In [2]:
# ============================================================
# Setup Path Project
# ============================================================

def find_project_root(start_path=None):
    """
    Mendeteksi root folder project berdasarkan keberadaan file hasil Tahap 05.
    """
    if start_path is None:
        start_path = Path.cwd().resolve()
    else:
        start_path = Path(start_path).resolve()

    candidate_paths = [start_path] + list(start_path.parents)

    for candidate in candidate_paths:
        expected_input = candidate / "data" / "processed" / "bertopic_document_topics_with_manual_reference.csv"
        if expected_input.exists():
            return candidate

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLE_DIR = REPORTS_DIR / "tables"
REPORT_FIGURE_DIR = REPORTS_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_MANUAL_CODING_FINAL = PROCESSED_DIR / "manual_coding_final.csv"
INPUT_BERTOPIC_DOCUMENT_TOPICS = PROCESSED_DIR / "bertopic_document_topics_with_manual_reference.csv"
INPUT_BERTOPIC_TOPIC_INFO = PROCESSED_DIR / "bertopic_topic_info.csv"
INPUT_BERTOPIC_TOPIC_KEYWORDS = PROCESSED_DIR / "bertopic_topic_keywords.csv"

print("Project path berhasil disiapkan.")
print(f"Current working directory           : {Path.cwd().resolve()}")
print(f"PROJECT_ROOT                        : {PROJECT_ROOT}")
print(f"INPUT_MANUAL_CODING_FINAL           : {INPUT_MANUAL_CODING_FINAL}")
print(f"INPUT_BERTOPIC_DOCUMENT_TOPICS      : {INPUT_BERTOPIC_DOCUMENT_TOPICS}")
print(f"INPUT_BERTOPIC_TOPIC_INFO           : {INPUT_BERTOPIC_TOPIC_INFO}")
print(f"INPUT_BERTOPIC_TOPIC_KEYWORDS       : {INPUT_BERTOPIC_TOPIC_KEYWORDS}")
print(f"REPORT_TABLE_DIR                    : {REPORT_TABLE_DIR}")
print(f"REPORT_FIGURE_DIR                   : {REPORT_FIGURE_DIR}")

Project path berhasil disiapkan.
Current working directory           : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\notebooks
PROJECT_ROOT                        : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
INPUT_MANUAL_CODING_FINAL           : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\manual_coding_final.csv
INPUT_BERTOPIC_DOCUMENT_TOPICS      : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_document_topics_with_manual_reference.csv
INPUT_BERTOPIC_TOPIC_INFO           : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_topic_info.csv
INPUT_BERTOPIC_TOPIC_KEYWORDS       : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_topic_keywords.csv
REPORT_TABLE_DIR                    : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MININ

## 3. Preflight Check Input

Cell ini memastikan seluruh file input utama tersedia.

In [3]:
# ============================================================
# Preflight Check Input
# ============================================================

required_input_files = {
    "manual_coding_final": INPUT_MANUAL_CODING_FINAL,
    "bertopic_document_topics_with_manual_reference": INPUT_BERTOPIC_DOCUMENT_TOPICS,
    "bertopic_topic_info": INPUT_BERTOPIC_TOPIC_INFO,
    "bertopic_topic_keywords": INPUT_BERTOPIC_TOPIC_KEYWORDS
}

missing_files = []

for name, file_path in required_input_files.items():
    if not file_path.exists():
        missing_files.append((name, file_path))

if missing_files:
    error_message = "File input berikut belum ditemukan:\n"
    for name, file_path in missing_files:
        error_message += f"- {name}: {file_path}\n"
    raise FileNotFoundError(error_message)

print("Seluruh file input Tahap 06 ditemukan.")
for name, file_path in required_input_files.items():
    print(f"- {name}: {file_path.stat().st_size:,} bytes")

Seluruh file input Tahap 06 ditemukan.
- manual_coding_final: 265,601 bytes
- bertopic_document_topics_with_manual_reference: 137,256 bytes
- bertopic_topic_info: 69,837 bytes
- bertopic_topic_keywords: 11,675 bytes


## 4. Membaca Dataset

Dataset yang dibaca:

1. Hasil Manual Coding final.
2. Hasil document-topic BERTopic.
3. Topic info BERTopic.
4. Topic keywords BERTopic.

In [4]:
# ============================================================
# Load Dataset
# ============================================================

manual_df = pd.read_csv(INPUT_MANUAL_CODING_FINAL)
bertopic_doc_df = pd.read_csv(INPUT_BERTOPIC_DOCUMENT_TOPICS)
topic_info_df = pd.read_csv(INPUT_BERTOPIC_TOPIC_INFO)
topic_keywords_df = pd.read_csv(INPUT_BERTOPIC_TOPIC_KEYWORDS)

print("Dataset berhasil dibaca.")
print(f"manual_df shape        : {manual_df.shape}")
print(f"bertopic_doc_df shape  : {bertopic_doc_df.shape}")
print(f"topic_info_df shape    : {topic_info_df.shape}")
print(f"topic_keywords_df shape: {topic_keywords_df.shape}")

print("\nPreview manual_df:")
display(manual_df.head())

print("\nPreview bertopic_doc_df:")
display(bertopic_doc_df.head())

Dataset berhasil dibaca.
manual_df shape        : (74, 34)
bertopic_doc_df shape  : (71, 24)
topic_info_df shape    : (15, 8)
topic_keywords_df shape: (150, 5)

Preview manual_df:


,chunk_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,source_url,source_domain,source_validation_status,...,matched_keywords_summary,suggestion_detail_json,manual_open_code_1,manual_open_code_2,manual_open_code_3,manual_axial_category,manual_selective_theme,coding_status,coder_notes,processed_stage04_at
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,9/8/2025,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC008=brics(6), multilateralism(1), cooperatio...","[{""open_code_id"": ""OC008"", ""open_code_label"": ...",Diplomasi multilateral dan kerja sama internas...,Anti-korupsi dan penegakan hukum,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,2026-06-11T22:11:05
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,9/8/2025,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,OC008=brics(2),"[{""open_code_id"": ""OC008"", ""open_code_label"": ...",Diplomasi multilateral dan kerja sama internas...,NaN,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,2026-06-11T22:11:05
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,1/7/2026,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC001=pangan(1), jagung(1), petani(1), pertani...","[{""open_code_id"": ""OC001"", ""open_code_label"": ...",Kedaulatan pangan dan swasembada,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,2026-06-11T22:11:05
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,1/7/2026,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC001=pangan(1), jagung(1), petani(2), pertani...","[{""open_code_id"": ""OC001"", ""open_code_label"": ...",Kedaulatan pangan dan swasembada,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,2026-06-11T22:11:05
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,1/7/2026,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,OC005=kesehatan(1),"[{""open_code_id"": ""OC005"", ""open_code_label"": ...","Kesehatan, gizi, dan perlindungan sosial",NaN,NaN,Pembangunan sumber daya manusia,Pembangunan manusia dan keadilan sosial,EXCLUDED,NaN,2026-06-11T22:11:05



Preview bertopic_doc_df:


,doc_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_order,text,chunk_word_count,...,manual_open_code_2,manual_open_code_3,manual_axial_category,manual_selective_theme,coding_status,coder_notes,bertopic_topic_id,bertopic_topic_probability,bertopic_topic_name,is_outlier_topic
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,1,Distinguished Leaders of BRICS. It is indeed a...,227,...,Anti-korupsi dan penegakan hukum,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,9,1.000000,9_brics_best_danantara_now,False
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,"We consider now, this is the time that BRICS m...",42,...,NaN,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,9,1.000000,9_brics_best_danantara_now,False
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,1,Bismillahirrahmanirrahim. Assalamu'alaikum war...,268,...,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,0,0.598132,0_bupati_hadir_menteri_hormati,False
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,2,"Yang saya hormati, para Dirut BUMN yang berken...",224,...,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,0,0.476953,0_bupati_hadir_menteri_hormati,False
4,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,4,"Walaupun selalu, kita selalu ingat saudara-sau...",222,...,Kedaulatan pangan dan swasembada,"Investasi, industrialisasi, dan pertumbuhan ek...",Identitas nasional dan kepemimpinan,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,10,1.000000,10_sudah_proyek_mengerti_juta,False


## 5. Validasi Kolom Wajib

Validasi kolom dilakukan secara eksplisit agar tidak ada asumsi struktur data.

In [5]:
# ============================================================
# Helper Validasi Kolom
# ============================================================

def require_columns(df, required_columns, df_name="DataFrame"):
    """
    Memastikan DataFrame memiliki kolom yang dibutuhkan.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{df_name} tidak memiliki kolom wajib: {missing_columns}. "
            f"Kolom tersedia: {list(df.columns)}"
        )


def clean_label(series, missing_label="MISSING"):
    """
    Membersihkan label kategorikal untuk analisis.
    """
    return (
        series
        .fillna(missing_label)
        .astype(str)
        .str.strip()
        .replace("", missing_label)
    )


REQUIRED_MANUAL_COLUMNS = [
    "chunk_id",
    "speech_id",
    "chunk_text",
    "manual_open_code_1",
    "manual_axial_category",
    "manual_selective_theme",
    "coding_status"
]

REQUIRED_BERTOPIC_DOC_COLUMNS = [
    "doc_id",
    "speech_id",
    "text",
    "bertopic_topic_id"
]

REQUIRED_TOPIC_INFO_COLUMNS = [
    "Topic",
    "Count",
    "Name"
]

REQUIRED_TOPIC_KEYWORD_COLUMNS = [
    "bertopic_topic_id",
    "keyword_rank",
    "keyword",
    "keyword_score"
]

require_columns(manual_df, REQUIRED_MANUAL_COLUMNS, "manual_df")
require_columns(bertopic_doc_df, REQUIRED_BERTOPIC_DOC_COLUMNS, "bertopic_doc_df")
require_columns(topic_info_df, REQUIRED_TOPIC_INFO_COLUMNS, "topic_info_df")
require_columns(topic_keywords_df, REQUIRED_TOPIC_KEYWORD_COLUMNS, "topic_keywords_df")

if manual_df["chunk_id"].duplicated().any():
    raise ValueError("Terdapat chunk_id duplikat pada manual_df.")

if bertopic_doc_df["doc_id"].duplicated().any():
    raise ValueError("Terdapat doc_id duplikat pada bertopic_doc_df.")

print("Validasi kolom wajib berhasil.")
print(f"Jumlah chunk manual      : {len(manual_df)}")
print(f"Jumlah dokumen BERTopic  : {len(bertopic_doc_df)}")
print(f"Jumlah topik BERTopic    : {topic_info_df['Topic'].nunique()}")

Validasi kolom wajib berhasil.
Jumlah chunk manual      : 74
Jumlah dokumen BERTopic  : 71
Jumlah topik BERTopic    : 15


## 6. Menyiapkan Data Komparatif

Manual Coding memiliki `chunk_id`, sedangkan BERTopic menggunakan `doc_id`.

Pada project ini, `doc_id` dibuat dari `chunk_id`, sehingga keduanya dapat digabungkan dengan:

```text
manual_df.chunk_id = bertopic_doc_df.doc_id
```

Chunk dengan `coding_status = EXCLUDED` biasanya tidak masuk ke BERTopic karena sudah dikeluarkan pada Tahap 05.

In [6]:
# ============================================================
# Prepare Comparative Dataset
# ============================================================

manual_for_merge_df = manual_df.copy()
manual_for_merge_df = manual_for_merge_df.rename(columns={"chunk_id": "doc_id"})

# Kolom manual yang dipakai untuk analisis.
manual_reference_columns = [
    "doc_id",
    "manual_open_code_1",
    "manual_open_code_2" if "manual_open_code_2" in manual_for_merge_df.columns else None,
    "manual_open_code_3" if "manual_open_code_3" in manual_for_merge_df.columns else None,
    "manual_axial_category",
    "manual_selective_theme",
    "coding_status",
    "coder_notes" if "coder_notes" in manual_for_merge_df.columns else None
]
manual_reference_columns = [col for col in manual_reference_columns if col is not None]

# Hapus kolom manual duplikat yang mungkin sudah ada di hasil BERTopic, agar merge bersih.
manual_cols_existing_in_bertopic = [
    col for col in manual_reference_columns
    if col != "doc_id" and col in bertopic_doc_df.columns
]

bertopic_base_df = bertopic_doc_df.drop(columns=manual_cols_existing_in_bertopic, errors="ignore").copy()

comparative_df = bertopic_base_df.merge(
    manual_for_merge_df[manual_reference_columns],
    on="doc_id",
    how="left",
    validate="one_to_one"
)

# Tambahkan label yang sudah dibersihkan.
comparative_df["manual_selective_theme_clean"] = clean_label(comparative_df["manual_selective_theme"])
comparative_df["manual_axial_category_clean"] = clean_label(comparative_df["manual_axial_category"])
comparative_df["manual_open_code_1_clean"] = clean_label(comparative_df["manual_open_code_1"])
comparative_df["coding_status_clean"] = clean_label(comparative_df["coding_status"])
comparative_df["bertopic_topic_id_clean"] = comparative_df["bertopic_topic_id"].astype(str)

# Join topic label dan keyword ringkas.
topic_name_map = dict(zip(topic_info_df["Topic"], topic_info_df["Name"]))

if "auto_topic_label" in topic_info_df.columns:
    topic_auto_label_map = dict(zip(topic_info_df["Topic"], topic_info_df["auto_topic_label"]))
else:
    topic_auto_label_map = {}

comparative_df["bertopic_topic_name_from_info"] = comparative_df["bertopic_topic_id"].map(topic_name_map)
comparative_df["bertopic_auto_topic_label"] = comparative_df["bertopic_topic_id"].map(topic_auto_label_map)

# Validasi merge.
missing_manual_after_merge = comparative_df["manual_selective_theme"].isna().sum()

print("Data komparatif berhasil dibuat.")
print(f"Jumlah baris comparative_df: {len(comparative_df)}")
print(f"Jumlah baris tanpa manual_selective_theme setelah merge: {missing_manual_after_merge}")

display(comparative_df.head())

Data komparatif berhasil dibuat.
Jumlah baris comparative_df: 71
Jumlah baris tanpa manual_selective_theme setelah merge: 0


,doc_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_order,text,chunk_word_count,...,manual_selective_theme,coding_status,coder_notes,manual_selective_theme_clean,manual_axial_category_clean,manual_open_code_1_clean,coding_status_clean,bertopic_topic_id_clean,bertopic_topic_name_from_info,bertopic_auto_topic_label
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,1,Distinguished Leaders of BRICS. It is indeed a...,227,...,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,"Diplomasi, perdamaian, dan keadilan global",Diplomasi dan tatanan global,Diplomasi multilateral dan kerja sama internas...,REVIEWED,9,9_brics_best_danantara_now,brics / best / danantara / now / biggest
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,"We consider now, this is the time that BRICS m...",42,...,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,"Diplomasi, perdamaian, dan keadilan global",Diplomasi dan tatanan global,Diplomasi multilateral dan kerja sama internas...,REVIEWED,9,9_brics_best_danantara_now,brics / best / danantara / now / biggest
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,1,Bismillahirrahmanirrahim. Assalamu'alaikum war...,268,...,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,Kedaulatan dan kemandirian nasional,Ketahanan nasional dan kedaulatan strategis,Kedaulatan pangan dan swasembada,REVIEWED,0,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,2,"Yang saya hormati, para Dirut BUMN yang berken...",224,...,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,Kedaulatan dan kemandirian nasional,Ketahanan nasional dan kedaulatan strategis,Kedaulatan pangan dan swasembada,REVIEWED,0,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota
4,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,4,"Walaupun selalu, kita selalu ingat saudara-sau...",222,...,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,Kedaulatan dan kemandirian nasional,Identitas nasional dan kepemimpinan,Persatuan nasional dan kepercayaan diri bangsa,REVIEWED,10,10_sudah_proyek_mengerti_juta,sudah / proyek / mengerti / juta / sembilan


## 7. Coverage Analysis

Analisis coverage digunakan untuk menjelaskan berapa banyak chunk yang:

1. Ada pada Manual Coding.
2. Digunakan oleh BERTopic.
3. Dikecualikan karena `EXCLUDED`.

In [7]:
# ============================================================
# Coverage Analysis
# ============================================================

manual_total_count = len(manual_df)
bertopic_modeled_count = len(bertopic_doc_df)

manual_reviewed_count = int((manual_df["coding_status"].fillna("").str.upper() == "REVIEWED").sum())
manual_excluded_count = int((manual_df["coding_status"].fillna("").str.upper() == "EXCLUDED").sum())
manual_needs_discussion_count = int((manual_df["coding_status"].fillna("").str.upper() == "NEEDS_DISCUSSION").sum())

manual_doc_ids = set(manual_df["chunk_id"].astype(str))
bertopic_doc_ids = set(bertopic_doc_df["doc_id"].astype(str))

manual_not_in_bertopic = sorted(list(manual_doc_ids - bertopic_doc_ids))
bertopic_not_in_manual = sorted(list(bertopic_doc_ids - manual_doc_ids))

coverage_records = [
    {
        "metric": "manual_total_chunk_count",
        "value": manual_total_count,
        "description": "Jumlah seluruh chunk pada manual_coding_final.csv."
    },
    {
        "metric": "manual_reviewed_chunk_count",
        "value": manual_reviewed_count,
        "description": "Jumlah chunk dengan coding_status REVIEWED."
    },
    {
        "metric": "manual_excluded_chunk_count",
        "value": manual_excluded_count,
        "description": "Jumlah chunk dengan coding_status EXCLUDED."
    },
    {
        "metric": "manual_needs_discussion_chunk_count",
        "value": manual_needs_discussion_count,
        "description": "Jumlah chunk dengan coding_status NEEDS_DISCUSSION."
    },
    {
        "metric": "bertopic_modeled_document_count",
        "value": bertopic_modeled_count,
        "description": "Jumlah dokumen chunk yang masuk ke BERTopic."
    },
    {
        "metric": "manual_not_in_bertopic_count",
        "value": len(manual_not_in_bertopic),
        "description": "Jumlah chunk manual yang tidak muncul dalam output BERTopic."
    },
    {
        "metric": "bertopic_not_in_manual_count",
        "value": len(bertopic_not_in_manual),
        "description": "Jumlah doc_id BERTopic yang tidak ada pada manual coding."
    }
]

coverage_report_df = pd.DataFrame(coverage_records)

display(coverage_report_df)

if manual_not_in_bertopic:
    print("Contoh manual chunk yang tidak masuk BERTopic:")
    print(manual_not_in_bertopic[:10])

if bertopic_not_in_manual:
    print("Contoh BERTopic doc_id yang tidak ada pada manual:")
    print(bertopic_not_in_manual[:10])

,metric,value,description
0,manual_total_chunk_count,74,Jumlah seluruh chunk pada manual_coding_final....
1,manual_reviewed_chunk_count,71,Jumlah chunk dengan coding_status REVIEWED.
2,manual_excluded_chunk_count,3,Jumlah chunk dengan coding_status EXCLUDED.
3,manual_needs_discussion_chunk_count,0,Jumlah chunk dengan coding_status NEEDS_DISCUS...
4,bertopic_modeled_document_count,71,Jumlah dokumen chunk yang masuk ke BERTopic.
5,manual_not_in_bertopic_count,3,Jumlah chunk manual yang tidak muncul dalam ou...
6,bertopic_not_in_manual_count,0,Jumlah doc_id BERTopic yang tidak ada pada man...


Contoh manual chunk yang tidak masuk BERTopic:
['SPCH_002_PANEN_RAYA_CHK_003', 'SPCH_004_PERESMIAN_166_SEKOLAH_CHK_004', 'SPCH_004_PERESMIAN_166_SEKOLAH_CHK_016']


## 8. Crosstab Manual Selective Theme vs BERTopic Topic

Tabulasi silang ini adalah inti dari analisis perbandingan.

Baris menunjukkan tema manual, sedangkan kolom menunjukkan topik BERTopic.

In [8]:
# ============================================================
# Crosstab Manual Theme vs BERTopic Topic
# ============================================================

comparative_analysis_df = comparative_df[
    comparative_df["coding_status_clean"].str.upper() != "EXCLUDED"
].copy()

if comparative_analysis_df.empty:
    raise ValueError("Tidak ada data untuk analisis setelah EXCLUDED dikeluarkan.")

comparative_crosstab_theme_topic_df = pd.crosstab(
    comparative_analysis_df["manual_selective_theme_clean"],
    comparative_analysis_df["bertopic_topic_id"]
)

comparative_crosstab_theme_topic_pct_df = comparative_crosstab_theme_topic_df.div(
    comparative_crosstab_theme_topic_df.sum(axis=1),
    axis=0
).fillna(0).round(4)

display(comparative_crosstab_theme_topic_df)
display(comparative_crosstab_theme_topic_pct_df)

bertopic_topic_id,-1,0,1,2,3,4,5,6,7,8,9,10,11,12,13
manual_selective_theme_clean,,,,,,,,,,,,,,,
"Diplomasi, perdamaian, dan keadilan global",0,2,2,1,0,0,0,0,4,0,2,0,0,0,1
Keberlanjutan lingkungan dan sumber daya,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
Kedaulatan dan kemandirian nasional,0,8,0,4,3,1,3,0,0,1,0,1,0,1,0
Pembangunan manusia dan keadilan sosial,1,2,2,1,2,3,1,4,0,3,0,1,3,0,1
Reformasi tata kelola dan penegakan hukum,0,1,2,0,0,0,0,0,0,0,0,1,0,1,1
Transformasi ekonomi dan pembangunan nasional,0,0,1,0,1,1,1,0,0,0,1,0,0,0,0


bertopic_topic_id,-1,0,1,2,3,4,5,6,7,8,9,10,11,12,13
manual_selective_theme_clean,,,,,,,,,,,,,,,
"Diplomasi, perdamaian, dan keadilan global",0.0000,0.1667,0.1667,0.0833,0.0000,0.0000,0.0000,0.0000,0.3333,0.0000,0.1667,0.0000,0.000,0.0000,0.0833
Keberlanjutan lingkungan dan sumber daya,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.5000,0.0000
Kedaulatan dan kemandirian nasional,0.0000,0.3636,0.0000,0.1818,0.1364,0.0455,0.1364,0.0000,0.0000,0.0455,0.0000,0.0455,0.000,0.0455,0.0000
Pembangunan manusia dan keadilan sosial,0.0417,0.0833,0.0833,0.0417,0.0833,0.1250,0.0417,0.1667,0.0000,0.1250,0.0000,0.0417,0.125,0.0000,0.0417
Reformasi tata kelola dan penegakan hukum,0.0000,0.1667,0.3333,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1667,0.000,0.1667,0.1667
Transformasi ekonomi dan pembangunan nasional,0.0000,0.0000,0.2000,0.0000,0.2000,0.2000,0.2000,0.0000,0.0000,0.0000,0.2000,0.0000,0.000,0.0000,0.0000


## 9. Mapping Dominan: BERTopic Topic ke Manual Theme

Mapping ini menunjukkan tema manual yang paling dominan pada setiap topik BERTopic.

Kolom penting:

- `dominant_manual_selective_theme`
- `dominant_theme_count`
- `topic_document_count`
- `topic_purity`

`topic_purity` semakin tinggi berarti satu topik BERTopic lebih konsisten dengan satu tema manual tertentu.

In [9]:
# ============================================================
# Dominant Mapping Topic to Manual Theme
# ============================================================

topic_theme_counts_df = (
    comparative_analysis_df
    .groupby(["bertopic_topic_id", "manual_selective_theme_clean"], dropna=False)
    .agg(document_count=("doc_id", "count"))
    .reset_index()
)

topic_total_counts_df = (
    comparative_analysis_df
    .groupby("bertopic_topic_id", dropna=False)
    .agg(topic_document_count=("doc_id", "count"))
    .reset_index()
)

idx = topic_theme_counts_df.groupby("bertopic_topic_id")["document_count"].idxmax()
dominant_topic_theme_df = topic_theme_counts_df.loc[idx].copy()

bertopic_topic_manual_theme_mapping_df = dominant_topic_theme_df.merge(
    topic_total_counts_df,
    on="bertopic_topic_id",
    how="left"
)

bertopic_topic_manual_theme_mapping_df = bertopic_topic_manual_theme_mapping_df.rename(columns={
    "manual_selective_theme_clean": "dominant_manual_selective_theme",
    "document_count": "dominant_theme_count"
})

bertopic_topic_manual_theme_mapping_df["topic_purity"] = (
    bertopic_topic_manual_theme_mapping_df["dominant_theme_count"] /
    bertopic_topic_manual_theme_mapping_df["topic_document_count"]
).round(4)

# Tambahkan nama topik dan label keyword.
topic_label_columns = ["Topic", "Name"]
if "auto_topic_label" in topic_info_df.columns:
    topic_label_columns.append("auto_topic_label")

bertopic_topic_manual_theme_mapping_df = bertopic_topic_manual_theme_mapping_df.merge(
    topic_info_df[topic_label_columns].rename(columns={"Topic": "bertopic_topic_id"}),
    on="bertopic_topic_id",
    how="left"
)

bertopic_topic_manual_theme_mapping_df = bertopic_topic_manual_theme_mapping_df.sort_values(
    ["bertopic_topic_id"]
).reset_index(drop=True)

display(bertopic_topic_manual_theme_mapping_df)

,bertopic_topic_id,dominant_manual_selective_theme,dominant_theme_count,topic_document_count,topic_purity,Name,auto_topic_label
0,-1,Keberlanjutan lingkungan dan sumber daya,1,2,0.5000,-1_melindungi_ancaman_melindungi ancaman_pertu...,Outlier / tidak terklaster
1,0,Kedaulatan dan kemandirian nasional,8,13,0.6154,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota
2,1,"Diplomasi, perdamaian, dan keadilan global",2,7,0.2857,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...
3,2,Kedaulatan dan kemandirian nasional,4,6,0.6667,2_energi_menghasilkan_kesulitan_hasilkan,energi / menghasilkan / kesulitan / hasilkan /...
4,3,Kedaulatan dan kemandirian nasional,3,6,0.5000,3_000_rice_years_000 murid,000 / rice / years / 000 murid / 83
5,4,Pembangunan manusia dan keadilan sosial,3,5,0.6000,4_kemerdekaan_dia_perang_mau,kemerdekaan / dia / perang / mau / enggak
6,5,Kedaulatan dan kemandirian nasional,3,5,0.6000,5_swasembada_amran_harga_tokoh,swasembada / amran / harga / tokoh / kulitnya
7,6,Pembangunan manusia dan keadilan sosial,4,4,1.0000,6_anak_orang tuamu_tuamu_son,anak / orang tuamu / tuamu / son / kau
8,7,"Diplomasi, perdamaian, dan keadilan global",4,4,1.0000,7_united nations_nations_united_peace,united nations / nations / united / peace / all
9,8,Pembangunan manusia dan keadilan sosial,3,4,0.7500,8_koperasi_meals_day_guru,koperasi / meals / day / guru / desember


## 10. Mapping Dominan: Manual Theme ke BERTopic Topic

Mapping ini menunjukkan topik BERTopic mana yang paling sering muncul untuk setiap tema manual.

In [10]:
# ============================================================
# Dominant Mapping Manual Theme to Topic
# ============================================================

theme_topic_counts_df = (
    comparative_analysis_df
    .groupby(["manual_selective_theme_clean", "bertopic_topic_id"], dropna=False)
    .agg(document_count=("doc_id", "count"))
    .reset_index()
)

theme_total_counts_df = (
    comparative_analysis_df
    .groupby("manual_selective_theme_clean", dropna=False)
    .agg(theme_document_count=("doc_id", "count"))
    .reset_index()
)

idx = theme_topic_counts_df.groupby("manual_selective_theme_clean")["document_count"].idxmax()
dominant_theme_topic_df = theme_topic_counts_df.loc[idx].copy()

manual_theme_bertopic_topic_mapping_df = dominant_theme_topic_df.merge(
    theme_total_counts_df,
    on="manual_selective_theme_clean",
    how="left"
)

manual_theme_bertopic_topic_mapping_df = manual_theme_bertopic_topic_mapping_df.rename(columns={
    "bertopic_topic_id": "dominant_bertopic_topic_id",
    "document_count": "dominant_topic_count"
})

manual_theme_bertopic_topic_mapping_df["theme_concentration"] = (
    manual_theme_bertopic_topic_mapping_df["dominant_topic_count"] /
    manual_theme_bertopic_topic_mapping_df["theme_document_count"]
).round(4)

manual_theme_bertopic_topic_mapping_df = manual_theme_bertopic_topic_mapping_df.merge(
    topic_info_df[topic_label_columns].rename(columns={
        "Topic": "dominant_bertopic_topic_id",
        "Name": "dominant_bertopic_topic_name"
    }),
    on="dominant_bertopic_topic_id",
    how="left"
)

manual_theme_bertopic_topic_mapping_df = manual_theme_bertopic_topic_mapping_df.sort_values(
    ["theme_document_count"],
    ascending=False
).reset_index(drop=True)

display(manual_theme_bertopic_topic_mapping_df)

,manual_selective_theme_clean,dominant_bertopic_topic_id,dominant_topic_count,theme_document_count,theme_concentration,dominant_bertopic_topic_name,auto_topic_label
0,Pembangunan manusia dan keadilan sosial,6,4,24,0.1667,6_anak_orang tuamu_tuamu_son,anak / orang tuamu / tuamu / son / kau
1,Kedaulatan dan kemandirian nasional,0,8,22,0.3636,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota
2,"Diplomasi, perdamaian, dan keadilan global",7,4,12,0.3333,7_united nations_nations_united_peace,united nations / nations / united / peace / all
3,Reformasi tata kelola dan penegakan hukum,1,2,6,0.3333,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...
4,Transformasi ekonomi dan pembangunan nasional,1,1,5,0.2000,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...
5,Keberlanjutan lingkungan dan sumber daya,-1,1,2,0.5000,-1_melindungi_ancaman_melindungi ancaman_pertu...,Outlier / tidak terklaster


## 11. Metrik Kesesuaian Clustering

Karena BERTopic adalah unsupervised clustering, metrik yang digunakan bukan accuracy.

Metrik yang digunakan:

1. **NMI**: hubungan informasi antara label manual dan cluster.
2. **AMI**: NMI yang disesuaikan terhadap peluang acak.
3. **Homogeneity**: satu cluster idealnya berisi dokumen dari satu tema manual.
4. **Completeness**: dokumen dari satu tema manual idealnya masuk cluster yang sama.
5. **V-measure**: harmonic mean antara homogeneity dan completeness.

Interpretasi umum:

- Mendekati 1 berarti relasi manual dan cluster semakin kuat.
- Mendekati 0 berarti relasi semakin lemah.

In [11]:
# ============================================================
# Clustering Agreement Metrics
# ============================================================

AUTO_INSTALL_MISSING_PACKAGES = True

def ensure_package(module_name, package_name):
    """
    Mengecek dan menginstal package jika belum tersedia.
    """
    if importlib.util.find_spec(module_name) is None:
        if AUTO_INSTALL_MISSING_PACKAGES:
            print(f"Package {package_name} belum tersedia. Instalasi dijalankan...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package_name])
        else:
            raise ImportError(f"Package {package_name} belum tersedia.")

ensure_package("sklearn", "scikit-learn")

from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score
)

manual_labels = comparative_analysis_df["manual_selective_theme_clean"].astype(str).tolist()
bertopic_labels = comparative_analysis_df["bertopic_topic_id"].astype(str).tolist()

nmi_score = normalized_mutual_info_score(manual_labels, bertopic_labels)
ami_score = adjusted_mutual_info_score(manual_labels, bertopic_labels)
homogeneity = homogeneity_score(manual_labels, bertopic_labels)
completeness = completeness_score(manual_labels, bertopic_labels)
v_measure = v_measure_score(manual_labels, bertopic_labels)

metrics_records = [
    {
        "metric": "normalized_mutual_information",
        "value": round(float(nmi_score), 4),
        "interpretation": "Mengukur hubungan informasi antara tema manual dan topik BERTopic."
    },
    {
        "metric": "adjusted_mutual_information",
        "value": round(float(ami_score), 4),
        "interpretation": "Mengukur hubungan informasi setelah dikoreksi terhadap peluang acak."
    },
    {
        "metric": "homogeneity",
        "value": round(float(homogeneity), 4),
        "interpretation": "Mengukur apakah setiap topik BERTopic cenderung berisi satu tema manual."
    },
    {
        "metric": "completeness",
        "value": round(float(completeness), 4),
        "interpretation": "Mengukur apakah dokumen dari tema manual yang sama cenderung masuk topik yang sama."
    },
    {
        "metric": "v_measure",
        "value": round(float(v_measure), 4),
        "interpretation": "Rata-rata harmonik antara homogeneity dan completeness."
    }
]

comparative_metrics_summary_df = pd.DataFrame(metrics_records)

display(comparative_metrics_summary_df)

,metric,value,interpretation
0,normalized_mutual_information,0.3340,Mengukur hubungan informasi antara tema manual...
1,adjusted_mutual_information,0.1263,Mengukur hubungan informasi setelah dikoreksi ...
2,homogeneity,0.4500,Mengukur apakah setiap topik BERTopic cenderun...
3,completeness,0.2656,Mengukur apakah dokumen dari tema manual yang ...
4,v_measure,0.3340,Rata-rata harmonik antara homogeneity dan comp...


## 12. Representative Documents by Topic

Tabel ini mengambil contoh dokumen paling representatif dari setiap topik BERTopic berdasarkan probabilitas topik tertinggi.

Jika kolom probabilitas tidak tersedia, dokumen dipilih berdasarkan urutan data.

In [12]:
# ============================================================
# Representative Documents by Topic
# ============================================================

probability_column = "bertopic_topic_probability"

representative_sort_columns = ["bertopic_topic_id"]
ascending_values = [True]

if probability_column in comparative_analysis_df.columns:
    representative_sort_columns.append(probability_column)
    ascending_values.append(False)

representative_documents_by_topic_df = (
    comparative_analysis_df
    .sort_values(representative_sort_columns, ascending=ascending_values)
    .groupby("bertopic_topic_id", dropna=False)
    .head(5)
    .copy()
)

representative_columns = [
    "doc_id",
    "speech_id",
    "file_name",
    "chunk_order",
    "bertopic_topic_id",
    "bertopic_topic_name" if "bertopic_topic_name" in representative_documents_by_topic_df.columns else None,
    "bertopic_topic_name_from_info" if "bertopic_topic_name_from_info" in representative_documents_by_topic_df.columns else None,
    "bertopic_topic_probability" if "bertopic_topic_probability" in representative_documents_by_topic_df.columns else None,
    "manual_open_code_1",
    "manual_axial_category",
    "manual_selective_theme",
    "text"
]

representative_columns = [col for col in representative_columns if col is not None and col in representative_documents_by_topic_df.columns]

representative_documents_by_topic_df = representative_documents_by_topic_df[representative_columns].copy()

display(representative_documents_by_topic_df.head(20))

,doc_id,speech_id,file_name,chunk_order,bertopic_topic_id,bertopic_topic_name,bertopic_topic_name_from_info,bertopic_topic_probability,manual_open_code_1,manual_axial_category,manual_selective_theme,text
67,SPCH_006_PBB_80_CHK_006,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,6,-1,-1_melindungi_ancaman_melindungi ancaman_pertu...,-1_melindungi_ancaman_melindungi ancaman_pertu...,0.099832,"Lingkungan, iklim, dan keberlanjutan",Ketahanan lingkungan dan keberlanjutan,Keberlanjutan lingkungan dan sumber daya,We aim to achieve net zero emission by 2060 an...
34,SPCH_004_PERESMIAN_166_SEKOLAH_CHK_006,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,6,-1,-1_melindungi_ancaman_melindungi ancaman_pertu...,-1_melindungi_ancaman_melindungi ancaman_pertu...,0.086919,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,"Nah, ini teori, tapi kenyataannya menetesnya k..."
18,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI_CHK_001,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-...,1,0,0_bupati_hadir_menteri_hormati,0_bupati_hadir_menteri_hormati,1.000000,Kedaulatan energi dan pengelolaan sumber daya,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,Bismillahirrahmanirrahim. Assalamu'alaikum war...
29,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI_CHK_012,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-...,12,0,0_bupati_hadir_menteri_hormati,0_bupati_hadir_menteri_hormati,1.000000,Kedaulatan energi dan pengelolaan sumber daya,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,Advanced technology ini dipimpin oleh seorang ...
30,SPCH_004_PERESMIAN_166_SEKOLAH_CHK_001,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,1,0,0_bupati_hadir_menteri_hormati,0_bupati_hadir_menteri_hormati,1.000000,Persatuan nasional dan kepercayaan diri bangsa,Identitas nasional dan kepemimpinan,Kedaulatan dan kemandirian nasional,"Bismillahirrahmanirrahim, Assalamu'alaikum war..."
32,SPCH_004_PERESMIAN_166_SEKOLAH_CHK_003,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,3,0,0_bupati_hadir_menteri_hormati,0_bupati_hadir_menteri_hormati,1.000000,Reformasi birokrasi dan efisiensi negara,Tata kelola pemerintahan dan rule of law,Reformasi tata kelola dan penegakan hukum,"Menko-Menko yang hadir tadi, ya, Menteri Koord..."
62,SPCH_006_PBB_80_CHK_001,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,1,0,0_bupati_hadir_menteri_hormati,0_bupati_hadir_menteri_hormati,1.000000,"Palestina, Gaza, dan keadilan global",Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global","His Excellency Mr. Antoni Guterres, Secretary ..."
45,SPCH_005_WORLD_ECONOMIC_FORUM_CHK_001,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,1,1,1_kampus_stability_growth_peace stability,1_kampus_stability_growth_peace stability,1.000000,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global","Distinguished President and CEO of the WEF, Mr..."
46,SPCH_005_WORLD_ECONOMIC_FORUM_CHK_002,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,2,1,1_kampus_stability_growth_peace stability,1_kampus_stability_growth_peace stability,1.000000,"Investasi, industrialisasi, dan pertumbuhan ek...",Transformasi ekonomi dan pembangunan produktif,Transformasi ekonomi dan pembangunan nasional,Peace and stability in my country did not happ...
63,SPCH_006_PBB_80_CHK_002,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,2,1,1_kampus_stability_growth_peace stability,1_kampus_stability_growth_peace stability,1.000000,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global","Human folly, fueled by fear, racism, hatred, o..."


## 13. Narrative Summary Table

Tabel ini membantu menyusun narasi laporan.

Setiap topik BERTopic dirangkum dengan:

1. Kata kunci utama.
2. Tema manual dominan.
3. Jumlah dokumen.
4. Purity.
5. Catatan interpretasi awal.

In [13]:
# ============================================================
# Comparative Narrative Summary
# ============================================================

top_keywords_by_topic_df = (
    topic_keywords_df
    .sort_values(["bertopic_topic_id", "keyword_rank"])
    .groupby("bertopic_topic_id")
    .head(8)
    .groupby("bertopic_topic_id")["keyword"]
    .apply(lambda values: ", ".join(values.astype(str).tolist()))
    .reset_index()
    .rename(columns={"keyword": "top_keywords"})
)

comparative_narrative_summary_df = bertopic_topic_manual_theme_mapping_df.merge(
    top_keywords_by_topic_df,
    on="bertopic_topic_id",
    how="left"
)

def build_interpretation_note(row):
    """
    Membuat catatan interpretasi awal berdasarkan mapping dominan.
    """
    topic_id = row["bertopic_topic_id"]
    theme = row["dominant_manual_selective_theme"]
    purity = row["topic_purity"]
    count = row["topic_document_count"]
    keywords = row.get("top_keywords", "")

    if topic_id == -1:
        return (
            f"Topik {topic_id} merupakan outlier/tidak terklaster. "
            f"Sebagian besar chunk pada kelompok ini terkait tema manual '{theme}', "
            f"namun perlu interpretasi hati-hati karena statusnya outlier."
        )

    return (
        f"Topik {topic_id} berisi {count} dokumen dan paling dominan terkait tema manual "
        f"'{theme}' dengan purity {purity}. Kata kunci utama: {keywords}."
    )

comparative_narrative_summary_df["interpretation_note"] = comparative_narrative_summary_df.apply(
    build_interpretation_note,
    axis=1
)

narrative_columns = [
    "bertopic_topic_id",
    "Name" if "Name" in comparative_narrative_summary_df.columns else None,
    "auto_topic_label" if "auto_topic_label" in comparative_narrative_summary_df.columns else None,
    "top_keywords",
    "dominant_manual_selective_theme",
    "dominant_theme_count",
    "topic_document_count",
    "topic_purity",
    "interpretation_note"
]

narrative_columns = [col for col in narrative_columns if col is not None]
comparative_narrative_summary_df = comparative_narrative_summary_df[narrative_columns].copy()

display(comparative_narrative_summary_df)

,bertopic_topic_id,Name,auto_topic_label,top_keywords,dominant_manual_selective_theme,dominant_theme_count,topic_document_count,topic_purity,interpretation_note
0,-1,-1_melindungi_ancaman_melindungi ancaman_pertu...,Outlier / tidak terklaster,"melindungi, ancaman, melindungi ancaman, pertu...",Keberlanjutan lingkungan dan sumber daya,1,2,0.5000,Topik -1 merupakan outlier/tidak terklaster. S...
1,0,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota,"bupati, hadir, menteri, hormati, wali kota, wa...",Kedaulatan dan kemandirian nasional,8,13,0.6154,Topik 0 berisi 13 dokumen dan paling dominan t...
2,1,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...,"kampus, stability, growth, peace stability, ov...","Diplomasi, perdamaian, dan keadilan global",2,7,0.2857,Topik 1 berisi 7 dokumen dan paling dominan te...
3,2,2_energi_menghasilkan_kesulitan_hasilkan,energi / menghasilkan / kesulitan / hasilkan /...,"energi, menghasilkan, kesulitan, hasilkan, ter...",Kedaulatan dan kemandirian nasional,4,6,0.6667,Topik 2 berisi 6 dokumen dan paling dominan te...
4,3,3_000_rice_years_000 murid,000 / rice / years / 000 murid / 83,"000, rice, years, 000 murid, 83, 83 000, sasar...",Kedaulatan dan kemandirian nasional,3,6,0.5000,Topik 3 berisi 6 dokumen dan paling dominan te...
5,4,4_kemerdekaan_dia_perang_mau,kemerdekaan / dia / perang / mau / enggak,"kemerdekaan, dia, perang, mau, enggak, rakyat,...",Pembangunan manusia dan keadilan sosial,3,5,0.6000,Topik 4 berisi 5 dokumen dan paling dominan te...
6,5,5_swasembada_amran_harga_tokoh,swasembada / amran / harga / tokoh / kulitnya,"swasembada, amran, harga, tokoh, kulitnya, jad...",Kedaulatan dan kemandirian nasional,3,5,0.6000,Topik 5 berisi 5 dokumen dan paling dominan te...
7,6,6_anak_orang tuamu_tuamu_son,anak / orang tuamu / tuamu / son / kau,"anak, orang tuamu, tuamu, son, kau, ya, anak a...",Pembangunan manusia dan keadilan sosial,4,4,1.0000,Topik 6 berisi 4 dokumen dan paling dominan te...
8,7,7_united nations_nations_united_peace,united nations / nations / united / peace / all,"united nations, nations, united, peace, all, t...","Diplomasi, perdamaian, dan keadilan global",4,4,1.0000,Topik 7 berisi 4 dokumen dan paling dominan te...
9,8,8_koperasi_meals_day_guru,koperasi / meals / day / guru / desember,"koperasi, meals, day, guru, desember, ribu, ko...",Pembangunan manusia dan keadilan sosial,3,4,0.7500,Topik 8 berisi 4 dokumen dan paling dominan te...


## 14. Visualisasi Perbandingan

Visualisasi yang dibuat:

1. Heatmap manual theme vs BERTopic topic.
2. Stacked bar manual theme dalam setiap topik BERTopic.
3. Distribusi jumlah dokumen manual theme dan topik BERTopic.

In [15]:
# ============================================================
# Comparative Visualizations
# ============================================================

AUTO_INSTALL_MISSING_PACKAGES = True

def ensure_package(module_name, package_name):
    if importlib.util.find_spec(module_name) is None:
        if AUTO_INSTALL_MISSING_PACKAGES:
            print(f"Package {package_name} belum tersedia. Instalasi dijalankan...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package_name])
        else:
            raise ImportError(f"Package {package_name} belum tersedia.")

ensure_package("plotly", "plotly")

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

heatmap_path = REPORT_FIGURE_DIR / "comparative_heatmap_manual_theme_vs_bertopic.html"
stacked_bar_path = REPORT_FIGURE_DIR / "comparative_stacked_bar_manual_theme_by_topic.html"
distribution_path = REPORT_FIGURE_DIR / "manual_vs_bertopic_distribution.html"

# ------------------------------------------------------------
# 1. Heatmap Manual Theme vs BERTopic Topic
# ------------------------------------------------------------

heatmap_fig = px.imshow(
    comparative_crosstab_theme_topic_df,
    text_auto=True,
    aspect="auto",
    title="Heatmap Manual Selective Theme vs BERTopic Topic",
    labels={
        "x": "BERTopic Topic ID",
        "y": "Manual Selective Theme",
        "color": "Jumlah Chunk"
    }
)

heatmap_fig.write_html(str(heatmap_path))


# ------------------------------------------------------------
# 2. Stacked Bar Manual Theme dalam Setiap Topik BERTopic
# ------------------------------------------------------------

stacked_source_df = (
    comparative_analysis_df
    .groupby(["bertopic_topic_id", "manual_selective_theme_clean"], dropna=False)
    .agg(document_count=("doc_id", "count"))
    .reset_index()
)

stacked_bar_fig = px.bar(
    stacked_source_df,
    x="bertopic_topic_id",
    y="document_count",
    color="manual_selective_theme_clean",
    title="Komposisi Manual Theme pada Setiap Topik BERTopic",
    labels={
        "bertopic_topic_id": "BERTopic Topic ID",
        "document_count": "Jumlah Chunk",
        "manual_selective_theme_clean": "Manual Selective Theme"
    }
)

stacked_bar_fig.write_html(str(stacked_bar_path))


# ------------------------------------------------------------
# 3. Perbandingan Distribusi Manual Theme dan BERTopic Topic
# ------------------------------------------------------------
# Catatan:
# Bagian ini dibuat ulang agar kompatibel dengan berbagai versi Pandas.
# Gunakan rename_axis("label").reset_index(name="document_count")
# agar kolomnya pasti: label, document_count.

manual_dist_df = (
    comparative_analysis_df["manual_selective_theme_clean"]
    .astype(str)
    .value_counts()
    .rename_axis("label")
    .reset_index(name="document_count")
)

manual_dist_df["type"] = "Manual Selective Theme"

topic_dist_df = (
    comparative_analysis_df["bertopic_topic_id"]
    .astype(str)
    .value_counts()
    .rename_axis("label")
    .reset_index(name="document_count")
)

topic_dist_df["type"] = "BERTopic Topic"

distribution_combined_df = pd.concat(
    [manual_dist_df, topic_dist_df],
    ignore_index=True
)

# Validasi kolom sebelum divisualisasikan
required_distribution_columns = ["label", "document_count", "type"]
missing_distribution_columns = [
    col for col in required_distribution_columns
    if col not in distribution_combined_df.columns
]

if missing_distribution_columns:
    raise ValueError(
        f"Kolom distribusi belum lengkap: {missing_distribution_columns}. "
        f"Kolom tersedia: {list(distribution_combined_df.columns)}"
    )

display(distribution_combined_df)

distribution_fig = px.bar(
    distribution_combined_df,
    x="label",
    y="document_count",
    color="type",
    barmode="group",
    title="Perbandingan Distribusi Manual Theme dan BERTopic Topic",
    labels={
        "label": "Label",
        "document_count": "Jumlah Chunk",
        "type": "Jenis Label"
    }
)

distribution_fig.update_layout(xaxis_tickangle=-45)
distribution_fig.write_html(str(distribution_path))

print("Visualisasi berhasil disimpan:")
print(f"- {heatmap_path}")
print(f"- {stacked_bar_path}")
print(f"- {distribution_path}")

,label,document_count,type
0,Pembangunan manusia dan keadilan sosial,24,Manual Selective Theme
1,Kedaulatan dan kemandirian nasional,22,Manual Selective Theme
2,"Diplomasi, perdamaian, dan keadilan global",12,Manual Selective Theme
3,Reformasi tata kelola dan penegakan hukum,6,Manual Selective Theme
4,Transformasi ekonomi dan pembangunan nasional,5,Manual Selective Theme
5,Keberlanjutan lingkungan dan sumber daya,2,Manual Selective Theme
6,0,13,BERTopic Topic
7,1,7,BERTopic Topic
8,2,6,BERTopic Topic
9,3,6,BERTopic Topic


Visualisasi berhasil disimpan:
- D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures\comparative_heatmap_manual_theme_vs_bertopic.html
- D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures\comparative_stacked_bar_manual_theme_by_topic.html
- D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures\manual_vs_bertopic_distribution.html


## 15. Validasi Akhir Sebelum Save

Cell ini memastikan output utama tidak kosong sebelum disimpan.

In [16]:
# ============================================================
# Cell 15 Final Validation
# ============================================================

outputs_to_validate = {
    "comparative_df": comparative_df,
    "comparative_analysis_df": comparative_analysis_df,
    "comparative_crosstab_theme_topic_df": comparative_crosstab_theme_topic_df,
    "bertopic_topic_manual_theme_mapping_df": bertopic_topic_manual_theme_mapping_df,
    "manual_theme_bertopic_topic_mapping_df": manual_theme_bertopic_topic_mapping_df,
    "comparative_metrics_summary_df": comparative_metrics_summary_df,
    "representative_documents_by_topic_df": representative_documents_by_topic_df,
    "comparative_narrative_summary_df": comparative_narrative_summary_df,
    "coverage_report_df": coverage_report_df
}

empty_outputs = [
    name for name, df in outputs_to_validate.items()
    if df is None or df.empty
]

if empty_outputs:
    raise ValueError(f"Output berikut kosong: {empty_outputs}")

print("Validasi akhir berhasil. Semua output utama siap disimpan.")

Validasi akhir berhasil. Semua output utama siap disimpan.


## 16. Menyimpan Output Tahap 06

Output disimpan ke folder:

```text
data/processed/
reports/tables/
reports/figures/
```

In [17]:
# ============================================================
# Save Output Tahap 06
# ============================================================

comparative_analysis_path = PROCESSED_DIR / "comparative_analysis_manual_vs_bertopic.csv"
comparative_crosstab_path = REPORT_TABLE_DIR / "comparative_crosstab_theme_topic.csv"
comparative_crosstab_pct_path = REPORT_TABLE_DIR / "comparative_crosstab_theme_topic_pct.csv"
bertopic_topic_manual_theme_mapping_path = REPORT_TABLE_DIR / "bertopic_topic_manual_theme_mapping.csv"
manual_theme_bertopic_topic_mapping_path = REPORT_TABLE_DIR / "manual_theme_bertopic_topic_mapping.csv"
comparative_metrics_summary_path = REPORT_TABLE_DIR / "comparative_metrics_summary.csv"
representative_documents_by_topic_path = REPORT_TABLE_DIR / "representative_documents_by_topic.csv"
comparative_narrative_summary_path = REPORT_TABLE_DIR / "comparative_narrative_summary.csv"
coverage_report_path = REPORT_TABLE_DIR / "comparative_coverage_report.csv"
stage06_output_manifest_path = REPORT_TABLE_DIR / "stage06_output_manifest.json"

comparative_df.to_csv(comparative_analysis_path, index=False, encoding="utf-8-sig")
comparative_crosstab_theme_topic_df.to_csv(comparative_crosstab_path, encoding="utf-8-sig")
comparative_crosstab_theme_topic_pct_df.to_csv(comparative_crosstab_pct_path, encoding="utf-8-sig")
bertopic_topic_manual_theme_mapping_df.to_csv(bertopic_topic_manual_theme_mapping_path, index=False, encoding="utf-8-sig")
manual_theme_bertopic_topic_mapping_df.to_csv(manual_theme_bertopic_topic_mapping_path, index=False, encoding="utf-8-sig")
comparative_metrics_summary_df.to_csv(comparative_metrics_summary_path, index=False, encoding="utf-8-sig")
representative_documents_by_topic_df.to_csv(representative_documents_by_topic_path, index=False, encoding="utf-8-sig")
comparative_narrative_summary_df.to_csv(comparative_narrative_summary_path, index=False, encoding="utf-8-sig")
coverage_report_df.to_csv(coverage_report_path, index=False, encoding="utf-8-sig")

stage06_output_manifest = {
    "stage": "06_comparative_analysis_manual_vs_bertopic",
    "input_files": {
        "manual_coding_final": str(INPUT_MANUAL_CODING_FINAL),
        "bertopic_document_topics_with_manual_reference": str(INPUT_BERTOPIC_DOCUMENT_TOPICS),
        "bertopic_topic_info": str(INPUT_BERTOPIC_TOPIC_INFO),
        "bertopic_topic_keywords": str(INPUT_BERTOPIC_TOPIC_KEYWORDS)
    },
    "outputs": {
        "comparative_analysis_manual_vs_bertopic": str(comparative_analysis_path),
        "comparative_crosstab_theme_topic": str(comparative_crosstab_path),
        "comparative_crosstab_theme_topic_pct": str(comparative_crosstab_pct_path),
        "bertopic_topic_manual_theme_mapping": str(bertopic_topic_manual_theme_mapping_path),
        "manual_theme_bertopic_topic_mapping": str(manual_theme_bertopic_topic_mapping_path),
        "comparative_metrics_summary": str(comparative_metrics_summary_path),
        "representative_documents_by_topic": str(representative_documents_by_topic_path),
        "comparative_narrative_summary": str(comparative_narrative_summary_path),
        "comparative_coverage_report": str(coverage_report_path),
        "comparative_heatmap_manual_theme_vs_bertopic": str(heatmap_path),
        "comparative_stacked_bar_manual_theme_by_topic": str(stacked_bar_path),
        "manual_vs_bertopic_distribution": str(distribution_path)
    },
    "manual_total_chunk_count": int(manual_total_count),
    "bertopic_modeled_document_count": int(bertopic_modeled_count),
    "comparative_analysis_row_count": int(len(comparative_analysis_df)),
    "manual_excluded_chunk_count": int(manual_excluded_count),
    "metrics": {
        "normalized_mutual_information": round(float(nmi_score), 4),
        "adjusted_mutual_information": round(float(ami_score), 4),
        "homogeneity": round(float(homogeneity), 4),
        "completeness": round(float(completeness), 4),
        "v_measure": round(float(v_measure), 4)
    },
    "created_at": datetime.now().isoformat(timespec="seconds")
}

stage06_output_manifest_path.write_text(
    json.dumps(stage06_output_manifest, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8"
)

print("Output Tahap 06 berhasil disimpan:")
print(f"1. {comparative_analysis_path}")
print(f"2. {comparative_crosstab_path}")
print(f"3. {comparative_crosstab_pct_path}")
print(f"4. {bertopic_topic_manual_theme_mapping_path}")
print(f"5. {manual_theme_bertopic_topic_mapping_path}")
print(f"6. {comparative_metrics_summary_path}")
print(f"7. {representative_documents_by_topic_path}")
print(f"8. {comparative_narrative_summary_path}")
print(f"9. {coverage_report_path}")
print(f"10. {stage06_output_manifest_path}")
print(f"11. {heatmap_path}")
print(f"12. {stacked_bar_path}")
print(f"13. {distribution_path}")

Output Tahap 06 berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\comparative_analysis_manual_vs_bertopic.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\comparative_crosstab_theme_topic.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\comparative_crosstab_theme_topic_pct.csv
4. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\bertopic_topic_manual_theme_mapping.csv
5. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\manual_theme_bertopic_topic_mapping.csv
6. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\comparative_metrics_summary.csv
7. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\representative_documents_by_topic.csv
8. D:\DATA-KAMIL\MATKUL\SEME

## 17. Preview Output Tahap 06

Preview ini digunakan untuk membaca hasil utama sebelum masuk ke tahap visualisasi dan laporan akhir.

In [18]:
# ============================================================
# Preview Output Tahap 06
# ============================================================

print("Coverage Report:")
display(coverage_report_df)

print("Comparative Metrics Summary:")
display(comparative_metrics_summary_df)

print("BERTopic Topic to Manual Theme Mapping:")
display(bertopic_topic_manual_theme_mapping_df)

print("Manual Theme to BERTopic Topic Mapping:")
display(manual_theme_bertopic_topic_mapping_df)

print("Narrative Summary:")
display(comparative_narrative_summary_df)

Coverage Report:


,metric,value,description
0,manual_total_chunk_count,74,Jumlah seluruh chunk pada manual_coding_final....
1,manual_reviewed_chunk_count,71,Jumlah chunk dengan coding_status REVIEWED.
2,manual_excluded_chunk_count,3,Jumlah chunk dengan coding_status EXCLUDED.
3,manual_needs_discussion_chunk_count,0,Jumlah chunk dengan coding_status NEEDS_DISCUS...
4,bertopic_modeled_document_count,71,Jumlah dokumen chunk yang masuk ke BERTopic.
5,manual_not_in_bertopic_count,3,Jumlah chunk manual yang tidak muncul dalam ou...
6,bertopic_not_in_manual_count,0,Jumlah doc_id BERTopic yang tidak ada pada man...


Comparative Metrics Summary:


,metric,value,interpretation
0,normalized_mutual_information,0.3340,Mengukur hubungan informasi antara tema manual...
1,adjusted_mutual_information,0.1263,Mengukur hubungan informasi setelah dikoreksi ...
2,homogeneity,0.4500,Mengukur apakah setiap topik BERTopic cenderun...
3,completeness,0.2656,Mengukur apakah dokumen dari tema manual yang ...
4,v_measure,0.3340,Rata-rata harmonik antara homogeneity dan comp...


BERTopic Topic to Manual Theme Mapping:


,bertopic_topic_id,dominant_manual_selective_theme,dominant_theme_count,topic_document_count,topic_purity,Name,auto_topic_label
0,-1,Keberlanjutan lingkungan dan sumber daya,1,2,0.5000,-1_melindungi_ancaman_melindungi ancaman_pertu...,Outlier / tidak terklaster
1,0,Kedaulatan dan kemandirian nasional,8,13,0.6154,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota
2,1,"Diplomasi, perdamaian, dan keadilan global",2,7,0.2857,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...
3,2,Kedaulatan dan kemandirian nasional,4,6,0.6667,2_energi_menghasilkan_kesulitan_hasilkan,energi / menghasilkan / kesulitan / hasilkan /...
4,3,Kedaulatan dan kemandirian nasional,3,6,0.5000,3_000_rice_years_000 murid,000 / rice / years / 000 murid / 83
5,4,Pembangunan manusia dan keadilan sosial,3,5,0.6000,4_kemerdekaan_dia_perang_mau,kemerdekaan / dia / perang / mau / enggak
6,5,Kedaulatan dan kemandirian nasional,3,5,0.6000,5_swasembada_amran_harga_tokoh,swasembada / amran / harga / tokoh / kulitnya
7,6,Pembangunan manusia dan keadilan sosial,4,4,1.0000,6_anak_orang tuamu_tuamu_son,anak / orang tuamu / tuamu / son / kau
8,7,"Diplomasi, perdamaian, dan keadilan global",4,4,1.0000,7_united nations_nations_united_peace,united nations / nations / united / peace / all
9,8,Pembangunan manusia dan keadilan sosial,3,4,0.7500,8_koperasi_meals_day_guru,koperasi / meals / day / guru / desember


Manual Theme to BERTopic Topic Mapping:


,manual_selective_theme_clean,dominant_bertopic_topic_id,dominant_topic_count,theme_document_count,theme_concentration,dominant_bertopic_topic_name,auto_topic_label
0,Pembangunan manusia dan keadilan sosial,6,4,24,0.1667,6_anak_orang tuamu_tuamu_son,anak / orang tuamu / tuamu / son / kau
1,Kedaulatan dan kemandirian nasional,0,8,22,0.3636,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota
2,"Diplomasi, perdamaian, dan keadilan global",7,4,12,0.3333,7_united nations_nations_united_peace,united nations / nations / united / peace / all
3,Reformasi tata kelola dan penegakan hukum,1,2,6,0.3333,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...
4,Transformasi ekonomi dan pembangunan nasional,1,1,5,0.2000,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...
5,Keberlanjutan lingkungan dan sumber daya,-1,1,2,0.5000,-1_melindungi_ancaman_melindungi ancaman_pertu...,Outlier / tidak terklaster


Narrative Summary:


,bertopic_topic_id,Name,auto_topic_label,top_keywords,dominant_manual_selective_theme,dominant_theme_count,topic_document_count,topic_purity,interpretation_note
0,-1,-1_melindungi_ancaman_melindungi ancaman_pertu...,Outlier / tidak terklaster,"melindungi, ancaman, melindungi ancaman, pertu...",Keberlanjutan lingkungan dan sumber daya,1,2,0.5000,Topik -1 merupakan outlier/tidak terklaster. S...
1,0,0_bupati_hadir_menteri_hormati,bupati / hadir / menteri / hormati / wali kota,"bupati, hadir, menteri, hormati, wali kota, wa...",Kedaulatan dan kemandirian nasional,8,13,0.6154,Topik 0 berisi 13 dokumen dan paling dominan t...
2,1,1_kampus_stability_growth_peace stability,kampus / stability / growth / peace stability ...,"kampus, stability, growth, peace stability, ov...","Diplomasi, perdamaian, dan keadilan global",2,7,0.2857,Topik 1 berisi 7 dokumen dan paling dominan te...
3,2,2_energi_menghasilkan_kesulitan_hasilkan,energi / menghasilkan / kesulitan / hasilkan /...,"energi, menghasilkan, kesulitan, hasilkan, ter...",Kedaulatan dan kemandirian nasional,4,6,0.6667,Topik 2 berisi 6 dokumen dan paling dominan te...
4,3,3_000_rice_years_000 murid,000 / rice / years / 000 murid / 83,"000, rice, years, 000 murid, 83, 83 000, sasar...",Kedaulatan dan kemandirian nasional,3,6,0.5000,Topik 3 berisi 6 dokumen dan paling dominan te...
5,4,4_kemerdekaan_dia_perang_mau,kemerdekaan / dia / perang / mau / enggak,"kemerdekaan, dia, perang, mau, enggak, rakyat,...",Pembangunan manusia dan keadilan sosial,3,5,0.6000,Topik 4 berisi 5 dokumen dan paling dominan te...
6,5,5_swasembada_amran_harga_tokoh,swasembada / amran / harga / tokoh / kulitnya,"swasembada, amran, harga, tokoh, kulitnya, jad...",Kedaulatan dan kemandirian nasional,3,5,0.6000,Topik 5 berisi 5 dokumen dan paling dominan te...
7,6,6_anak_orang tuamu_tuamu_son,anak / orang tuamu / tuamu / son / kau,"anak, orang tuamu, tuamu, son, kau, ya, anak a...",Pembangunan manusia dan keadilan sosial,4,4,1.0000,Topik 6 berisi 4 dokumen dan paling dominan te...
8,7,7_united nations_nations_united_peace,united nations / nations / united / peace / all,"united nations, nations, united, peace, all, t...","Diplomasi, perdamaian, dan keadilan global",4,4,1.0000,Topik 7 berisi 4 dokumen dan paling dominan te...
9,8,8_koperasi_meals_day_guru,koperasi / meals / day / guru / desember,"koperasi, meals, day, guru, desember, ribu, ko...",Pembangunan manusia dan keadilan sosial,3,4,0.7500,Topik 8 berisi 4 dokumen dan paling dominan te...


## 18. Interpretasi Hasil Tahap 06

Interpretasi umum yang dapat digunakan dalam laporan:

1. **Manual Coding** memberikan interpretasi tematik berbasis pembacaan peneliti.
2. **BERTopic** memberikan pengelompokan komputasional berbasis embedding dan clustering.
3. Jika satu topic BERTopic memiliki `topic_purity` tinggi, maka topic tersebut relatif konsisten dengan satu tema manual.
4. Jika satu tema manual tersebar ke banyak topic BERTopic, maka tema tersebut memiliki variasi bahasa atau konteks yang luas.
5. Jika topic `-1` muncul, itu menunjukkan dokumen yang tidak masuk cluster utama.
6. NMI, AMI, Homogeneity, Completeness, dan V-measure tidak perlu bernilai sempurna karena Manual Coding dan BERTopic bekerja dengan prinsip berbeda.

## Tahap Berikutnya

Tahap berikutnya adalah:

```text
Tahap 07 — Visualization, Interpretation, dan Final Reporting
```

Tahap 07 akan menyusun visualisasi final, tabel interpretasi, dan narasi akademik untuk laporan/presentasi.